# AA3 · Examen del Parcial III · Problema 1: CardioRitmo Wearable
**Equipo:** (nombre o número del equipo)
**Integrantes:** (nombres)

CardioRitmo quiere llevar su detector de latidos anormales a un **parche de bajo costo**. Pide:
- en la prueba, con umbral 0.5, **sensibilidad ≥ 95 %** y **especificidad ≥ 95 %**;
- un modelo de **2,000 parámetros o menos** (la Conv1D de la P7 tiene 13,057);
- una **página pública** donde su director clasifique, desde su celular, los latidos de la prueba.

**Cómo trabajar:** sigan las secciones en orden y **no cambien sus títulos**. Pueden agregar las celdas que necesiten. Al final, ejecuten todo de arriba abajo y guarden el cuaderno con sus salidas. El código de las prácticas que les sirve: la P7 (Optuna) y la P8 (ONNX, INT8 y la página).

In [ ]:
# Instalación (Colab no trae Optuna ni ONNX). No importen keras antes de la celda de Datos: ahí se elige PyTorch como motor.
import importlib.util, subprocess, sys
for paquete, modulo in [("optuna", "optuna"), ("onnx", "onnx"), ("onnxruntime", "onnxruntime"), ("onnxscript", "onnxscript")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
print("Listo")

## 1. Datos
La celda de partida del enunciado, **sin cambios**: carga ECG5000 con la división del examen del P2, la red de la P7 (`crear_modelo`), las métricas y el `INTENTO_0`. Si Keras ya se había importado con otro motor, reinicien la sesión antes de correrla.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"      # como en la P8: la exportación a ONNX usa PyTorch
import numpy as np, pandas as pd, keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

ecg = pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv", header=None)
X = ecg.iloc[:, :-1].to_numpy("float32")[..., np.newaxis]      # (4998, 140, 1)
y = (1 - ecg.iloc[:, -1]).to_numpy("int64")                     # 1 = ANORMAL
X_resto, X_prueba, y_resto, y_prueba = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_ent, X_val, y_ent, y_val = train_test_split(X_resto, y_resto, test_size=0.2, stratify=y_resto, random_state=42)

def crear_modelo(bloques, filtros, kernel, dropout):          # la familia de la P7
    entrada = keras.Input((140, 1), name="latido")
    x = entrada
    for i in range(bloques):
        x = layers.Conv1D(filtros * 2 ** i, kernel, padding="same", activation="relu")(x)
        x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(dropout)(layers.GlobalAveragePooling1D()(x))
    return keras.Model(entrada, layers.Dense(1, activation="sigmoid")(x), name="latidos")

def metricas(p, yy):                                           # p = probabilidades; umbral 0.5
    tn, fp, fn, tp = confusion_matrix(yy, (p >= 0.5).astype(int)).ravel()
    return {"sensibilidad": tp / (tp + fn), "especificidad": tn / (tn + fp), "auc": roc_auc_score(yy, p)}

INTENTO_0 = {"bloques": 3, "filtros": 16, "kernel": 5, "dropout": 0.2, "lr": 1e-3}   # la Conv1D de la P7

In [ ]:
# ✏️ Muestren cuántos latidos hay en entrenamiento, validación y prueba (y qué fracción son anormales)


## 2. Intento 0
Entrenen la red de `INTENTO_0` como en la P7: Adam con la tasa de `INTENTO_0`, `binary_crossentropy`, 40 épocas, lotes de 64 y `EarlyStopping(patience=6, restore_best_weights=True)`. Reporten sus **parámetros** y sus métricas **de validación** (todavía no la prueba).

In [ ]:
# ✏️ Intento 0


## 3. Búsqueda con Optuna
Un estudio con **dos objetivos**, como en la búsqueda de arquitecturas de la P7:
- `directions=["maximize", "minimize"]`;
- objetivo 1: la **peor** de sensibilidad y especificidad en validación, `min(sens, esp)`;
- objetivo 2: el número de parámetros, `modelo.count_params()`;
- al menos **30 intentos**, con `INTENTO_0` como primero (`enqueue_trial`);
- espacio con `bloques`, `filtros`, `kernel`, `dropout` y `lr`, **con 4 entre las opciones de filtros**.

Después: la gráfica del **frente de Pareto** (parámetros en el eje x, en escala logarítmica; `min(sens, esp)` en el eje y), la importancia de los hiperparámetros para el objetivo 1 (`optuna.importance.get_param_importances(estudio, target=lambda t: t.values[0])`) y `estudio.trials_dataframe().to_csv("optuna_ecg.csv", index=False)`.

Con redes tan pequeñas, los 30 intentos tardan unos 12 minutos.

In [ ]:
# ✏️ El estudio de Optuna


In [ ]:
# ✏️ Frente de Pareto, importancia y optuna_ecg.csv


## 4. Elección y prueba
**La regla, antes de ver la prueba:** entre los intentos del frente de Pareto con `min(sens, esp)` ≥ 0.97 en validación, el de **menos parámetros**. Si ninguno llega a 0.97, el de mayor `min(sens, esp)`.

Después usen la prueba **una sola vez**: el intento 0 y el elegido, con parámetros, sensibilidad, especificidad y AUC. **No usen `X_prueba` antes de esta sección.**

In [ ]:
# ✏️ Aplicar la regla


In [ ]:
# ✏️ La prueba, una sola vez: intento 0 contra el elegido


## 5. Edge: ONNX e INT8
Con el código de la P8:
1. exporten a ONNX el intento 0 y el elegido, y verifiquen que el ONNX del elegido da lo mismo que Keras (diferencia máxima);
2. cuanticen el elegido a INT8 **dinámica** y **estática** (calibración con `X_ent[:200]`);
3. midan cada versión: KB del archivo, milisegundos por latido con **1 hilo** (mediana) y métricas en la prueba;
4. decidan qué versión va al celular y por qué.

Guarden la tabla como `resultados_ecg.csv`, con las columnas `modelo`, `version` (`float32`, `INT8 dinámica` o `INT8 estática`), `parametros`, `kb`, `ms`, `sensibilidad`, `especificidad`, `auc` (de 0 a 1), `cumple` y `elegido` (`True` solo en la versión que va al celular).

En `ecg/modelo/` del repositorio van `latidos_float32.onnx` (el elegido), `latidos_int8_dinamica.onnx` y `latidos_int8_estatica.onnx`.

In [ ]:
# ✏️ Exportar a ONNX y verificar contra Keras


In [ ]:
# ✏️ Cuantizar a INT8 (dinámica y estática)


In [ ]:
# ✏️ Medir cada versión, decidir y guardar resultados_ecg.csv


## 6. Página en el celular
La plantilla ya está en la carpeta `docs/` del repositorio. En `docs/latidos/` agreguen cuatro archivos:
- `latidos_float32.onnx`;
- `latidos_int8.onnx`: la INT8 **estática** (ONNX Runtime Web, el motor de la página, no ejecuta la dinámica);
- `config.json`, como el de la P8;
- `latidos.json`, con los **primeros 200 latidos** de `X_prueba` y sus etiquetas, como en la P8.

Después: *Settings → Pages → Deploy from a branch → main → /docs*. Abran `https://USUARIO.github.io/aa3-examen-p3-equipo-N/latidos/` en el celular, presionen «Clasificar todos y medir» con las dos versiones y guarden la captura como `ecg/captura_celular.png` (o `.jpg`).

In [ ]:
# ✏️ Los cuatro archivos de docs/latidos/ (en Colab, descárguenlos y súbanlos al repositorio)


**URL de la página:** (péguenla aquí y en el README)

## 7. Respuestas
Una celda por pregunta. Usen sus números.

**R1.** ¿Cuántas veces más pequeño quedó su modelo que el intento 0, y qué perdió en sensibilidad y especificidad de prueba?

*Escriban aquí su respuesta, con sus números.*

**R2.** Según Optuna, ¿qué hiperparámetro importó más para el desempeño? ¿Por qué creen que fue ese?

*Escriban aquí su respuesta, con sus números.*

**R3.** ¿Valió la pena el INT8 en un modelo de este tamaño? Respondan con su tabla (KB, ms y métricas) y con lo que midieron en el celular.

*Escriban aquí su respuesta, con sus números.*

**R4.** ¿Por qué la regla de elección se aplica con la validación y la prueba se usa una sola vez? ¿Qué pasaría si eligieran con la prueba?

*Escriban aquí su respuesta, con sus números.*

**R5.** ¿Con qué pacientes funcionaría este modelo? Den una limitación de los datos y el aviso que le darían al director de CardioRitmo.

*Escriban aquí su respuesta, con sus números.*